In [1]:
import pandas as pd
import numpy as np
import os
import re

In [2]:
FILE_PATH = "Superstore_Pandas_Project.csv"

In [3]:
def load_dataset(file_path):
    """
    Automatically loads CSV or Excel files.
    """

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")

    extension = os.path.splitext(file_path)[1].lower()

    if extension == ".csv":
        df = pd.read_csv(file_path)

    elif extension in [".xlsx", ".xls"]:
        df = pd.read_excel(file_path)

    else:
        raise ValueError(
            "Unsupported file format. Use CSV, XLSX or XLS."
        )

    return df

In [4]:
df = load_dataset(FILE_PATH)

print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Dataset loaded successfully!
Rows: 10294
Columns: 21


In [5]:
def clean_column_names(df):

    df = df.copy()

    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
        .str.replace(r"[^\w]", "", regex=True)
    )

    return df


df = clean_column_names(df)

print("\nCleaned Columns:")
print(df.columns.tolist())


Cleaned Columns:
['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode', 'customer_id', 'customer_name', 'segment', 'country', 'city', 'state', 'postal_code', 'region', 'product_id', 'category', 'sub_category', 'product_name', 'sales', 'quantity', 'discount', 'profit']


In [6]:
duplicate_count = df.duplicated().sum()

print("\nDuplicate Rows:", duplicate_count)

df = df.drop_duplicates()


Duplicate Rows: 130


In [7]:
def find_column(df, possible_names):

    for name in possible_names:

        name = name.lower()

        if name in df.columns:
            return name

    return None

In [8]:
date_col = find_column(
    df,
    [
        "order_date",
        "date",
        "sales_date",
        "transaction_date",
        "invoice_date",
        "orderdate"
    ]
)

In [9]:
sales_col = find_column(
    df,
    [
        "sales",
        "revenue",
        "amount",
        "total_sales",
        "total_revenue",
        "price"
    ]
)

In [10]:
profit_col = find_column(
    df,
    [
        "profit",
        "net_profit",
        "gross_profit"
    ]
)

In [11]:
quantity_col = find_column(
    df,
    [
        "quantity",
        "qty",
        "units",
        "units_sold"
    ]
)


In [12]:
category_col = find_column(
    df,
    [
        "category",
        "product_category",
        "product_type"
    ]
)

In [13]:
subcategory_col = find_column(
    df,
    [
        "sub_category",
        "subcategory",
        "sub_category_name"
    ]
)


In [14]:
region_col = find_column(
    df,
    [
        "region",
        "area",
        "zone",
        "territory"
    ]
)

In [15]:
product_col = find_column(
    df,
    [
        "product_name",
        "product",
        "item",
        "item_name"
    ]
)

In [16]:
customer_col = find_column(
    df,
    [
        "customer_id",
        "customerid",
        "customer"
    ]
)


In [17]:
order_col = find_column(
    df,
    [
        "order_id",
        "orderid",
        "transaction_id",
        "invoice_id"
    ]
)


In [18]:
print("\nDetected Columns")

print("Date:", date_col)
print("Sales:", sales_col)
print("Profit:", profit_col)
print("Quantity:", quantity_col)
print("Category:", category_col)
print("Sub-category:", subcategory_col)
print("Region:", region_col)
print("Product:", product_col)
print("Customer:", customer_col)
print("Order:", order_col)


Detected Columns
Date: order_date
Sales: sales
Profit: profit
Quantity: quantity
Category: category
Sub-category: sub_category
Region: region
Product: product_name
Customer: customer_id
Order: order_id


In [19]:
numeric_columns = [
    sales_col,
    profit_col,
    quantity_col
]

In [20]:
for col in numeric_columns:

    if col is not None:

        df[col] = (
            df[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.replace("$", "", regex=False)
            .str.replace("₹", "", regex=False)
            .str.replace("€", "", regex=False)
            .str.strip()
        )

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

In [21]:
if date_col is not None:

    df[date_col] = pd.to_datetime(
        df[date_col],
        errors="coerce"
    )

    print("\nDate conversion completed.")


Date conversion completed.


In [22]:
missing_report = (
    df.isnull()
    .sum()
    .reset_index()
)

missing_report.columns = [
    "Column",
    "Missing_Values"
]

missing_report["Missing_Percentage"] = (
    missing_report["Missing_Values"]
    / len(df)
    * 100
)

print("\nMissing Value Report:")
print(missing_report)



Missing Value Report:
           Column  Missing_Values  Missing_Percentage
0          row_id               0            0.000000
1        order_id               0            0.000000
2      order_date               0            0.000000
3       ship_date               0            0.000000
4       ship_mode             308            3.030303
5     customer_id               0            0.000000
6   customer_name               0            0.000000
7         segment               0            0.000000
8         country               0            0.000000
9            city               0            0.000000
10          state               0            0.000000
11    postal_code             511            5.027548
12         region               0            0.000000
13     product_id               0            0.000000
14       category             505            4.968516
15   sub_category               0            0.000000
16   product_name               0            0.000000
17   

In [23]:
for col in df.select_dtypes(
    include=np.number
).columns:

    df[col] = df[col].fillna(
        df[col].median()
    )

In [24]:
for col in df.select_dtypes(
    include="object"
).columns:

    df[col] = df[col].fillna(
        "Unknown"
    )

In [25]:
for col in df.select_dtypes(
    include="object"
).columns:

    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
    )

In [26]:
if date_col is not None:

    df["year"] = df[date_col].dt.year

    df["month"] = df[date_col].dt.month

    df["month_name"] = (
        df[date_col]
        .dt.strftime("%B")
    )

    df["quarter"] = (
        df[date_col]
        .dt.quarter
    )

    df["year_month"] = (
        df[date_col]
        .dt.to_period("M")
        .astype(str)
    )

In [27]:
if sales_col is not None and profit_col is not None:

    df["profit_margin"] = np.where(
        df[sales_col] != 0,
        (df[profit_col] / df[sales_col]) * 100,
        0
    )

In [28]:
total_sales = (
    df[sales_col].sum()
    if sales_col
    else np.nan
)

total_profit = (
    df[profit_col].sum()
    if profit_col
    else np.nan
)

total_quantity = (
    df[quantity_col].sum()
    if quantity_col
    else np.nan
)

In [29]:
if order_col:

    total_orders = df[order_col].nunique()

else:

    total_orders = len(df)


if customer_col:

    total_customers = (
        df[customer_col]
        .nunique()
    )

else:

    total_customers = np.nan

In [30]:
if (
    sales_col is not None
    and total_orders != 0
):

    average_order_value = (
        total_sales / total_orders
    )

else:

    average_order_value = np.nan


In [31]:
if (
    sales_col is not None
    and profit_col is not None
    and total_sales != 0
):

    overall_profit_margin = (
        total_profit
        / total_sales
        * 100
    )

else:

    overall_profit_margin = np.nan


In [32]:
kpi = pd.DataFrame({

    "KPI": [
        "Total Revenue",
        "Total Profit",
        "Total Quantity",
        "Total Orders",
        "Total Customers",
        "Average Order Value",
        "Profit Margin"
    ],

    "Value": [
        total_sales,
        total_profit,
        total_quantity,
        total_orders,
        total_customers,
        average_order_value,
        overall_profit_margin
    ]

})

In [33]:
print("\n==============================")
print("BUSINESS KPIs")
print("==============================")

print(kpi)


BUSINESS KPIs
                   KPI         Value
0        Total Revenue  2.219004e+07
1         Total Profit  2.772836e+05
2       Total Quantity  3.795600e+04
3         Total Orders  5.009000e+03
4      Total Customers  7.930000e+02
5  Average Order Value  4.430033e+03
6        Profit Margin  1.249586e+00


In [34]:
if date_col and sales_col:

    monthly_sales = (
        df.groupby("year_month")[sales_col]
        .sum()
        .reset_index()
    )

    monthly_sales = monthly_sales.sort_values(
        "year_month"
    )

else:

    monthly_sales = pd.DataFrame()

In [35]:
if category_col and sales_col:

    category_analysis = (
        df.groupby(category_col)
        .agg(
            Revenue=(sales_col, "sum")
        )
        .reset_index()
    )

    if profit_col:

        profit_data = (
            df.groupby(category_col)[profit_col]
            .sum()
            .reset_index()
        )

        category_analysis = category_analysis.merge(
            profit_data,
            on=category_col,
            how="left"
        )
    if quantity_col:
    
            quantity_data = (
                df.groupby(category_col)[quantity_col]
                .sum()
                .reset_index()
            )
    
            category_analysis = category_analysis.merge(
                quantity_data,
                on=category_col,
                how="left"
            )

else:

    category_analysis = pd.DataFrame()

In [36]:
if region_col and sales_col:

    region_analysis = (
        df.groupby(region_col)
        .agg(
            Revenue=(sales_col, "sum")
        )
        .reset_index()
    )

    if profit_col:

        profit_data = (
            df.groupby(region_col)[profit_col]
            .sum()
            .reset_index()
        )

        region_analysis = region_analysis.merge(
            profit_data,
            on=region_col,
            how="left"
        )
else:

    region_analysis = pd.DataFrame()

In [37]:
if product_col and sales_col:

    top_products = (
        df.groupby(product_col)[sales_col]
        .sum()
        .sort_values(
            ascending=False
        )
        .head(10)
        .reset_index()
    )

else:

    top_products = pd.DataFrame()

In [38]:
if product_col and profit_col:

    top_profit_products = (
        df.groupby(product_col)[profit_col]
        .sum()
        .sort_values(
            ascending=False
        )
        .head(10)
        .reset_index()
    )

else:

    top_profit_products = pd.DataFrame()


In [39]:
if product_col and profit_col:

    loss_products = (
        df.groupby(product_col)[profit_col]
        .sum()
        .sort_values()
        .head(10)
        .reset_index()
    )

else:

    loss_products = pd.DataFrame()


In [40]:
print("\n==============================")
print("TOP PRODUCTS")
print("==============================")

print(top_products)


print("\n==============================")
print("CATEGORY ANALYSIS")
print("==============================")

print(category_analysis)


print("\n==============================")
print("REGION ANALYSIS")
print("==============================")

print(region_analysis)


print("\n==============================")
print("LOSS-MAKING PRODUCTS")
print("==============================")

print(loss_products)


TOP PRODUCTS
                                        product_name         sales
0                                  Samsung Galaxy S4  1.005132e+06
1           Plantronics Audio 478 Stereo USB Headset  1.001499e+06
2   Sauder Forest Hills Library, Woodland Oak Finish  1.001423e+06
3                                  Samsung Rugby III  1.001187e+06
4                                 Maxell 4.7GB DVD-R  1.001089e+06
5  Bush Westfield Collection Bookcases, Fully Ass...  1.000959e+06
6                  Fellowes Recycled Storage Drawers  1.000954e+06
7                              Surelock Post Binders  1.000431e+06
8                  Apple EarPods with Remote and Mic  1.000167e+06
9                            Crayola Colored Pencils  1.000162e+06

CATEGORY ANALYSIS
          category       Revenue       profit  quantity
0        Furniture  2.575805e+06   15523.9067      6511
1       Furnitures  2.394855e+04    -179.0117       302
2  Offcie Supplies  1.038242e+06    8029.2486       983
3  Off

In [41]:
os.makedirs(
    "output",
    exist_ok=True
)

df.to_csv(
    "output/cleaned_sales_data.csv",
    index=False
)


In [42]:
with pd.ExcelWriter(
    "output/sales_analysis.xlsx",
    engine="openpyxl"
) as writer:

    df.to_excel(
        writer,
        sheet_name="Cleaned Data",
        index=False
    )

    kpi.to_excel(
        writer,
        sheet_name="KPIs",
        index=False
    )

    if not monthly_sales.empty:

        monthly_sales.to_excel(
            writer,
            sheet_name="Monthly Sales",
            index=False
        )
    if not category_analysis.empty:

        category_analysis.to_excel(
            writer,
            sheet_name="Category Analysis",
            index=False
        )

    if not region_analysis.empty:

        region_analysis.to_excel(
            writer,
            sheet_name="Region Analysis",
            index=False
        )

    if not top_products.empty:

        top_products.to_excel(
            writer,
            sheet_name="Top Products",
            index=False
        )
    if not loss_products.empty:

        loss_products.to_excel(
            writer,
            sheet_name="Loss Products",
            index=False
        )

In [43]:
print("\n================================")
print("ANALYSIS COMPLETED SUCCESSFULLY")
print("================================")

print("\nFiles generated inside 'output' folder:")
print("1. cleaned_sales_data.csv")
print("2. sales_analysis.xlsx")


ANALYSIS COMPLETED SUCCESSFULLY

Files generated inside 'output' folder:
1. cleaned_sales_data.csv
2. sales_analysis.xlsx


In [1]:
import pandas as pd

df = pd.read_csv("Superstore_Pandas_Project.csv")

# Apni cleaning ke baad
df.to_excel(
    "cleaned_sales_powerbi1.xlsx",
    index=False,
    engine="openpyxl"
)

print("Excel file created successfully!")

Excel file created successfully!
